In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm


In [ ]:
df_MIRAGE = pd.read_parquet(
    "/Users/pacomelefebvre/Desktop/HCSP2/MIRAGE/data/MIRAGE_grouped_data.parquet"
).drop(columns="Quantity")


## Market share

In [ ]:
df_sum = (df_MIRAGE.copy()[["MIRAGE_code", "Year", "k$"]]
          .groupby(by=["MIRAGE_code", "Year"], as_index=False)
          .sum()
          .rename(columns={"k$": "k$_tot"}))
df_MIRAGE_full = pd.merge(left=df_MIRAGE, right=df_sum, on=["MIRAGE_code", "Year"], how="left")

In [ ]:
df_MIRAGE_full["market_share"] = df_MIRAGE_full["k$"] / df_MIRAGE_full["k$_tot"]
df_MIRAGE_full.to_csv("/Users/pacomelefebvre/Desktop/HCSP2/MIRAGE/data/market_share_all.csv")

## Market share progression

In [ ]:
df_MIRAGE["Year"].unique()

In [ ]:
# Regroupement des montant par secteur
agg = df_MIRAGE.groupby(["Exporter", "MIRAGE_code", "Year"], as_index=False)["k$"].sum()

# Rotation pour avoir une colonne par année
wide = agg.pivot(index=["Exporter", "MIRAGE_code"], columns="Year", values="k$")

years = list(range(2019, 2024))
result = pd.DataFrame(index=wide.index)
for y1, y2 in zip(years[:-1], years[1:]):
    result[f"évolution_{y1}_{y2}"] = wide[y2] / wide[y1] - 1 # ratio année N+1 et année N
result["évolution_2019_2023"] = wide[2023] / wide[2019] - 1 # ratio 2023 / 2019

result = result.reset_index().rename(columns={"Exporter": "Pays"})

In [ ]:
result.to_csv("/Users/pacomelefebvre/Desktop/HCSP2/MIRAGE/data/market_share_evolution.csv")